# Adim 3 — Spark ile Veri Isleme ve Delta Lake

Bu notebook iki islemi kapsar:

1. **Batch Ingestion (Egitim Verisi)**: Ocak 1-21 verisini dogrudan parquet'ten okuyup Delta Lake Silver'a yazar (~%75)
2. **Streaming Ingestion (Canli Veri)**: Ocak 22+ verisini Kafka'dan okuyup Delta Lake Silver'a yazar (~%25)

```
Tam Parquet (~2.9M kayit)
    |
    +-- Ocak 1-21 (~%75) --> [Batch] --> Delta Bronze --> Delta Silver --> EDA/Feature/ML
    |
    +-- Ocak 22-31 (~%25) --> [Kafka Streaming] --> Delta Stream Bronze --> Delta Stream Silver
```

## 3.1 Sabitler ve Yapilandirma

In [1]:
import os

# Kafka
KAFKA_BOOTSTRAP_SERVERS = "kafka:29092"
KAFKA_TOPIC = "taxi_trips"

# Veri yollari
DATA_DIR = "/home/jovyan/work/data"
PARQUET_FILE = os.path.join(DATA_DIR, "yellow_tripdata_2024-01.parquet")

# Delta Lake yollari
DELTA_BASE = os.path.join(DATA_DIR, "delta", "taxi_trips")
CHECKPOINT_BASE = os.path.join(DATA_DIR, "delta", "_checkpoints")

BRONZE_PATH = os.path.join(DELTA_BASE, "bronze")
SILVER_PATH = os.path.join(DELTA_BASE, "silver")
BRONZE_CHECKPOINT = os.path.join(CHECKPOINT_BASE, "bronze")
SILVER_CHECKPOINT = os.path.join(CHECKPOINT_BASE, "silver")

# Tarih siniri
TRAIN_CUTOFF = "2024-01-22"

print(f"Parquet: {PARQUET_FILE}")
print(f"Bronze:  {BRONZE_PATH}")
print(f"Silver:  {SILVER_PATH}")
print(f"Egitim verisi: < {TRAIN_CUTOFF}")
print(f"Stream verisi: >= {TRAIN_CUTOFF}")

Parquet: /home/jovyan/work/data/yellow_tripdata_2024-01.parquet
Bronze:  /home/jovyan/work/data/delta/taxi_trips/bronze
Silver:  /home/jovyan/work/data/delta/taxi_trips/silver
Egitim verisi: < 2024-01-22
Stream verisi: >= 2024-01-22


## 3.2 Spark Session Olusturma

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC-Taxi-Pipeline") \
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.1.0,"
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} baslatildi.")

Spark 3.5.0 baslatildi.


---

# BOLUM A: Batch Ingestion (Egitim Verisi)

Ocak 1-21 arasindaki veriyi dogrudan parquet'ten okuyup Bronze ve Silver Delta Lake tablolarina yaziyoruz.
Bu veri **sabit** ve **ML egitiminde** kullanilacak.

## A.1 Parquet'ten Egitim Verisini Oku

In [3]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

# Tum veriyi oku
raw_df = spark.read.parquet(PARQUET_FILE)
total_count = raw_df.count()
print(f"Toplam kayit: {total_count:,}")

# Egitim verisi: pickup < 22 Ocak
train_raw = raw_df.filter(col("tpep_pickup_datetime") < TRAIN_CUTOFF)
train_count = train_raw.count()

# Stream verisi: pickup >= 22 Ocak
stream_count = total_count - train_count

print(f"\nEgitim verisi (< {TRAIN_CUTOFF}): {train_count:,} ({train_count*100/total_count:.1f}%)")
print(f"Stream verisi (>= {TRAIN_CUTOFF}): {stream_count:,} ({stream_count*100/total_count:.1f}%)")

Toplam kayit: 2,964,624

Egitim verisi (< 2024-01-22): 1,966,586 (66.3%)
Stream verisi (>= 2024-01-22): 998,038 (33.7%)


## A.2 Bronze Katman (Ham Veri)

In [4]:
# Egitim verisini Bronze'a yaz (ham haliyle)
train_raw.write.format("delta").mode("overwrite").save(BRONZE_PATH)

bronze_count = spark.read.format("delta").load(BRONZE_PATH).count()
print(f"Bronze'a yazildi: {bronze_count:,} kayit")
print(f"Yol: {BRONZE_PATH}")

Bronze'a yazildi: 1,966,586 kayit
Yol: /home/jovyan/work/data/delta/taxi_trips/bronze


## A.3 Silver Katman (Temizlenmis Veri)

Temizlik kurallari:
- `trip_distance > 0`
- `total_amount > 0` ve `< 500`
- `passenger_count > 0`
- Null datetime kayitlar filtrelenir
- Duplike kayitlar kaldirilir

In [5]:
# Bronze'dan oku
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

# Temizlik
silver_df = bronze_df \
    .filter(
        (col("trip_distance") > 0) &
        (col("total_amount") > 0) &
        (col("total_amount") < 500) &
        (col("passenger_count") > 0) &
        (col("tpep_pickup_datetime").isNotNull()) &
        (col("tpep_dropoff_datetime").isNotNull())
    ) \
    .dropDuplicates(["tpep_pickup_datetime", "PULocationID", "DOLocationID", "total_amount"])

# Silver'a yaz
silver_df.write.format("delta").mode("overwrite").save(SILVER_PATH)

silver_count = spark.read.format("delta").load(SILVER_PATH).count()
print(f"Bronze: {bronze_count:,} -> Silver: {silver_count:,} kayit")
print(f"Filtrelenen: {bronze_count - silver_count:,} kayit ({(bronze_count - silver_count)*100/bronze_count:.1f}%)")

Bronze: 1,966,586 -> Silver: 1,800,870 kayit
Filtrelenen: 165,716 kayit (8.4%)


## A.4 Silver Dogrulama

In [6]:
silver_check = spark.read.format("delta").load(SILVER_PATH)
silver_check.printSchema()
silver_check.show(5, truncate=False)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------

In [7]:
# Tarih araliklarini kontrol et
from pyspark.sql.functions import min as spark_min, max as spark_max

silver_check.select(
    spark_min("tpep_pickup_datetime").alias("min_pickup"),
    spark_max("tpep_pickup_datetime").alias("max_pickup"),
    spark_min("tpep_dropoff_datetime").alias("min_dropoff"),
    spark_max("tpep_dropoff_datetime").alias("max_dropoff")
).show(truncate=False)

print("Egitim verisi sadece Ocak 1-21 arasini icermeli.")

+-------------------+-------------------+-------------------+-------------------+
|min_pickup         |max_pickup         |min_dropoff        |max_dropoff        |
+-------------------+-------------------+-------------------+-------------------+
|2002-12-31 22:59:39|2024-01-21 23:59:58|2002-12-31 23:05:41|2024-01-22 21:41:25|
+-------------------+-------------------+-------------------+-------------------+

Egitim verisi sadece Ocak 1-21 arasini icermeli.


---

# BOLUM B: Streaming Ingestion (Kafka)

Ocak 22+ verisi Kafka Producer tarafindan stream ediliyor.
Bu bolumde Spark Structured Streaming ile Kafka'dan okuyup ayri bir Delta tablosuna yaziyoruz.

**Not:** Bu bolum streaming yetenegi gostermek icindir. ML egitimi Bolum A'daki sabit veriden yapilir.

## B.1 Kafka Mesaj Semasi

In [8]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType,
    IntegerType
)

taxi_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", StringType(), True),
    StructField("tpep_dropoff_datetime", StringType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True)
])

print("Kafka mesaj semasi tanimlandi.")

Kafka mesaj semasi tanimlandi.


## B.2 Kafka'dan Stream Okuma

In [9]:
from pyspark.sql.functions import from_json

# Streaming veri icin ayri yollar
STREAM_BRONZE = os.path.join(DELTA_BASE, "stream_bronze")
STREAM_SILVER = os.path.join(DELTA_BASE, "stream_silver")
STREAM_BRONZE_CHECKPOINT = os.path.join(CHECKPOINT_BASE, "stream_bronze")
STREAM_SILVER_CHECKPOINT = os.path.join(CHECKPOINT_BASE, "stream_silver")

# Kafka'dan oku
raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "earliest") \
    .load()

# JSON parse
stream_bronze_df = raw_stream.select(
    from_json(
        col("value").cast("string"),
        taxi_schema
    ).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

print("Kafka stream okunuyor...")

Kafka stream okunuyor...


In [10]:
# Stream Bronze'u yaz
stream_bronze_query = stream_bronze_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", STREAM_BRONZE_CHECKPOINT) \
    .outputMode("append") \
    .start(STREAM_BRONZE)

print(f"Stream Bronze yaziliyor: {STREAM_BRONZE}")

Stream Bronze yaziliyor: /home/jovyan/work/data/delta/taxi_trips/stream_bronze


In [11]:
# Stream Silver (temizlenmis)
from pyspark.sql.types import TimestampType as TT

stream_silver_df = stream_bronze_df \
    .filter(
        (col("trip_distance") > 0) &
        (col("total_amount") > 0) &
        (col("total_amount") < 500) &
        (col("passenger_count") > 0) &
        (col("tpep_pickup_datetime").isNotNull()) &
        (col("tpep_dropoff_datetime").isNotNull())
    ) \
    .withColumn("tpep_pickup_datetime", col("tpep_pickup_datetime").cast(TT())) \
    .withColumn("tpep_dropoff_datetime", col("tpep_dropoff_datetime").cast(TT())) \
    .dropDuplicates(["tpep_pickup_datetime", "PULocationID", "DOLocationID", "total_amount"])

stream_silver_query = stream_silver_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", STREAM_SILVER_CHECKPOINT) \
    .outputMode("append") \
    .start(STREAM_SILVER)

print(f"Stream Silver yaziliyor: {STREAM_SILVER}")

Stream Silver yaziliyor: /home/jovyan/work/data/delta/taxi_trips/stream_silver


## B.3 Stream Durumunu Kontrol

In [12]:
import time
time.sleep(15)

for s in spark.streams.active:
    print(f"Stream: {s.id}")
    print(f"  Status: {s.status}")
    if s.recentProgress:
        latest = s.recentProgress[-1]
        print(f"  Rows: {latest.get('numInputRows', 'N/A')}")
    print()

Stream: e6acdc52-038a-4696-8dfb-40d660751407
  Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
  Rows: 2400

Stream: 8fafd2f7-ebef-4024-b9b7-760ff5ef5ccc
  Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
  Rows: 2400



In [13]:
# Stream tablolarini kontrol et
try:
    sb = spark.read.format("delta").load(STREAM_BRONZE)
    print(f"Stream Bronze: {sb.count():,} kayit")
except:
    print("Stream Bronze henuz olusturulmadi.")

try:
    ss = spark.read.format("delta").load(STREAM_SILVER)
    print(f"Stream Silver: {ss.count():,} kayit")
except:
    print("Stream Silver henuz olusturulmadi.")

Stream Bronze: 1,326,000 kayit
Stream Silver: 351,865 kayit


## B.4 Stream'leri Durdurma

In [ ]:
for s in spark.streams.active:
    print(f"Durduruluyor: {s.id}")
    s.stop()

print("Tum stream'ler durduruldu.")

---

## Ozet

| Katman | Veri | Yontem | Kullanim |
|--------|------|--------|----------|
| Bronze | Ocak 1-21 | Batch (parquet) | Egitim verisi |
| Silver | Ocak 1-21 | Batch (temizlenmis) | EDA, Feature Eng., ML |
| Stream Bronze | Ocak 22+ | Kafka Streaming | Streaming demo |
| Stream Silver | Ocak 22+ | Kafka Streaming (temizlenmis) | Streaming demo |